# 03 — Gold Layer
Read Silver Parquet. Build all aggregation tables that feed the dashboard.
Save each as a Gold Parquet. No raw data access, no cleaning here.

## Load Silver Parquet

In [1]:
import pandas as pd
import numpy as np

df       = pd.read_parquet("silver.parquet")
resolved = df.dropna(subset=["resolution_time_hours"])
csat_df  = df.dropna(subset=["csat_score"])

print("Total tickets:   ", len(df))
print("Resolved tickets:", len(resolved))
print("CSAT responses:  ", len(csat_df))

Total tickets:    7817
Resolved tickets: 4667
CSAT responses:   5475


## Resolution time by priority

In [2]:
priority_resolution = (
    resolved
    .groupby("priority")["resolution_time_hours"]
    .median()
    .reindex(["urgent", "high", "medium", "low"])
    .reset_index()
)
priority_resolution.columns = ["priority", "median_hours"]
print(priority_resolution)
priority_resolution.to_parquet("gold_priority_resolution.parquet", index=False)
print("Saved: gold_priority_resolution.parquet")

  priority  median_hours
0   urgent          4.64
1     high         15.27
2   medium         29.96
3      low         44.73
Saved: gold_priority_resolution.parquet


## Resolution time by SLA plan

In [3]:
sla_resolution = (
    resolved
    .groupby("sla_plan")["resolution_time_hours"]
    .median()
    .sort_values()
    .reset_index()
)
sla_resolution.columns = ["sla_plan", "median_hours"]
print(sla_resolution)
sla_resolution.to_parquet("gold_sla_resolution.parquet", index=False)
print("Saved: gold_sla_resolution.parquet")

   sla_plan  median_hours
0  platinum        25.915
1      gold        28.850
2  standard        29.870
Saved: gold_sla_resolution.parquet


## Monthly ticket volume

In [4]:
monthly = (
    df.groupby("year_month")
    .agg(n_tickets=("ticket_id", "count"), avg_csat=("csat_score", "mean"))
    .reset_index()
    .sort_values("year_month")
)
print(monthly.head())
monthly.to_parquet("gold_monthly.parquet", index=False)
print("Saved: gold_monthly.parquet")

  year_month  n_tickets  avg_csat
0    2022-01        158  3.391304
1    2022-02        150  3.176471
2    2022-03        169  3.209677
3    2022-04        146  3.053097
4    2022-05        164  3.324786
Saved: gold_monthly.parquet


## Status breakdown

In [5]:
status_breakdown = df["status"].value_counts().reset_index()
status_breakdown.columns = ["status", "count"]
print(status_breakdown)
status_breakdown.to_parquet("gold_status.parquet", index=False)
print("Saved: gold_status.parquet")

             status  count
0          resolved   3940
1       in_progress   1547
2           on_hold    830
3              open    773
4  closed_no_action    727
Saved: gold_status.parquet


## CSAT by SLA plan

In [6]:
csat_by_sla = (
    csat_df.groupby("sla_plan")["csat_score"]
    .mean()
    .sort_values()
    .reset_index()
)
csat_by_sla.columns = ["sla_plan", "csat_score"]
print(csat_by_sla)
csat_by_sla.to_parquet("gold_csat_sla.parquet", index=False)
print("Saved: gold_csat_sla.parquet")

   sla_plan  csat_score
0      gold    3.189156
1  standard    3.211875
2  platinum    3.220974
Saved: gold_csat_sla.parquet


## Resolution time by region and SLA

In [7]:
region_sla = (
    resolved
    .groupby(["region", "sla_plan"])["resolution_time_hours"]
    .median()
    .unstack()
    .reset_index()
)
print(region_sla)
region_sla.to_parquet("gold_region_sla.parquet", index=False)
print("Saved: gold_region_sla.parquet")

sla_plan region    gold  platinum  standard
0          APAC  30.760    28.240    29.970
1            EU  26.750    24.815    28.420
2         LATAM  31.000    23.280    28.665
3           MEA  28.035    32.710    32.430
4            NA  30.785    21.930    27.640
Saved: gold_region_sla.parquet


## Ticket volume by channel and issue type

In [8]:
channel_counts = df["channel"].value_counts().reset_index()
channel_counts.columns = ["channel", "count"]

issue_counts = df["issue_type"].value_counts().reset_index()
issue_counts.columns = ["issue_type", "count"]

print(channel_counts)
print(issue_counts)
channel_counts.to_parquet("gold_channel.parquet", index=False)
issue_counts.to_parquet("gold_issue_type.parquet", index=False)
print("Saved: gold_channel.parquet, gold_issue_type.parquet")

            channel  count
0             email   1640
1  phone_transcript   1595
2          web_form   1551
3            in_app   1524
4              chat   1507
         issue_type  count
0            how_to   1052
1  security_concern   1018
2               bug    967
3       performance    966
4   feature_request    961
5    account_access    961
6   billing_problem    958
7             other    934
Saved: gold_channel.parquet, gold_issue_type.parquet


## Ticket volume by product area

In [9]:
product_counts = df["product_area"].value_counts().reset_index()
product_counts.columns = ["product_area", "count"]
print(product_counts)
product_counts.to_parquet("gold_product_area.parquet", index=False)
print("Saved: gold_product_area.parquet")

          product_area  count
0  analytics_dashboard   1159
1           mobile_app   1128
2      api_integration   1125
3              billing   1109
4        notifications   1108
5           login_auth   1097
6          data_export   1091
Saved: gold_product_area.parquet


## CSAT heatmap (segment × region)

In [10]:
csat_heatmap = (
    csat_df
    .groupby(["customer_segment", "region"])["csat_score"]
    .mean()
    .unstack()
    .reset_index()
)
print(csat_heatmap)
csat_heatmap.to_parquet("gold_csat_heatmap.parquet", index=False)
print("Saved: gold_csat_heatmap.parquet")

region customer_segment      APAC        EU     LATAM       MEA        NA
0             education  3.238095  3.078067  3.389286  3.293878  3.300000
1            enterprise  3.250000  3.205224  3.090566  3.293893  3.153846
2            individual  3.177536  3.056034  3.094077  3.340164  3.300000
3            non_profit  3.173228  3.153257  3.266423  3.117424  2.909091
4        small_business  3.328063  3.233577  3.186441  3.195219  3.057143
Saved: gold_csat_heatmap.parquet


## Issue type stats — CSAT, reopen rate, resolution time

In [11]:
issue_stats = (
    df.groupby("issue_type")
    .agg(
        avg_csat        = ("csat_score",            "mean"),
        reopen_rate     = ("reopened",               "mean"),
        median_res_time = ("resolution_time_hours",  "median"),
    )
    .sort_values("avg_csat")
    .reset_index()
)
print(issue_stats)
issue_stats.to_parquet("gold_issue_stats.parquet", index=False)
print("Saved: gold_issue_stats.parquet")

         issue_type  avg_csat  reopen_rate  median_res_time
0   billing_problem  2.757037     0.045929           29.470
1    account_access  2.773196     0.055151           26.030
2  security_concern  2.795518     0.049116           31.730
3       performance  2.861862     0.054865           26.590
4             other  3.409712     0.051392           30.055
5               bug  3.469897     0.055843           28.805
6   feature_request  3.762195     0.052029           31.790
7            how_to  3.797315     0.051331           30.060
Saved: gold_issue_stats.parquet


## Sentiment distribution

In [12]:
sentiment_counts = (
    df["customer_sentiment"]
    .value_counts()
    .reindex(["very_negative", "negative", "neutral", "positive", "very_positive"])
    .reset_index()
)
sentiment_counts.columns = ["customer_sentiment", "count"]
print(sentiment_counts)
sentiment_counts.to_parquet("gold_sentiment.parquet", index=False)
print("Saved: gold_sentiment.parquet")

  customer_sentiment  count
0      very_negative   1317
1           negative   2056
2            neutral   2458
3           positive   1316
4      very_positive    670
Saved: gold_sentiment.parquet


## CSAT score distribution (1–5 bar chart)

In [13]:
csat_dist = (
    csat_df["csat_score"]
    .value_counts()
    .sort_index()
    .reset_index()
)
csat_dist.columns = ["csat_score", "count"]
print(csat_dist)
csat_dist.to_parquet("gold_csat_dist.parquet", index=False)
print("Saved: gold_csat_dist.parquet")

   csat_score  count
0         1.0    461
1         2.0   1173
2         3.0   1549
3         4.0   1361
4         5.0    931
Saved: gold_csat_dist.parquet


## Headline metrics

In [14]:
metrics = {
    "total_tickets":          len(df),
    "resolution_rate_pct":    round(df["is_resolved"].mean() * 100, 1),
    "median_resolution_hours":round(resolved["resolution_time_hours"].median(), 1),
    "mean_resolution_hours":  round(resolved["resolution_time_hours"].mean(), 1),
    "reopen_rate_pct":        round(df["reopened"].mean() * 100, 1),
    "csat_response_rate_pct": round(len(csat_df) / len(df) * 100, 1),
    "avg_csat":               round(csat_df["csat_score"].mean(), 2),
    "unresolved_backlog":     int((~df["is_resolved"]).sum()),
}
for k, v in metrics.items():
    print(f"  {k}: {v}")

pd.DataFrame([metrics]).to_parquet("gold_metrics.parquet", index=False)
print("Saved: gold_metrics.parquet")

  total_tickets: 7817
  resolution_rate_pct: 59.7
  median_resolution_hours: 29.3
  mean_resolution_hours: 43.4
  reopen_rate_pct: 5.2
  csat_response_rate_pct: 70.0
  avg_csat: 3.21
  unresolved_backlog: 3150
Saved: gold_metrics.parquet


## Confirm all Gold files

In [15]:
import os
gold_files = sorted([f for f in os.listdir(".") if f.startswith("gold_")])
print(f"{len(gold_files)} Gold files saved:")
for f in gold_files:
    print(" ", f)

14 Gold files saved:
  gold_channel.parquet
  gold_csat_dist.parquet
  gold_csat_heatmap.parquet
  gold_csat_sla.parquet
  gold_issue_stats.parquet
  gold_issue_type.parquet
  gold_metrics.parquet
  gold_monthly.parquet
  gold_priority_resolution.parquet
  gold_product_area.parquet
  gold_region_sla.parquet
  gold_sentiment.parquet
  gold_sla_resolution.parquet
  gold_status.parquet
